In [1]:
import sys
import os
import urllib.request

## **Dependency Check**

In [2]:
def check_dependencies(path):
    missing = []
    try:
        import cv2
    except ImportError:
        missing.append("opencv-python")
    try:
        import numpy
    except ImportError:
        missing.append("numpy")
    if path == 1:
        try:
            import pytesseract
        except ImportError:
            missing.append("pytesseract")
    if missing:
        print(f"\n Missing packages: {', '.join(missing)}")
        print(f"  Run: pip install {' '.join(missing)}")
        if path == 1:
            print("Also install Tesseract OCR engine:")
            print("Windows : https://github.com/UB-Mannheim/tesseract/wiki")
            print("Linux   : sudo apt install tesseract-ocr")
            print("macOS   : brew install tesseract")
        sys.exit(1)

## **Model Downloader**

In [3]:
MOBILENET_PROTOTXT = "MobileNetSSD_deploy.prototxt"
MOBILENET_MODEL    = "MobileNetSSD_deploy.caffemodel"
 
PROTOTXT_URL  = "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/deploy.prototxt"
MODEL_URL     = "https://drive.google.com/uc?export=download&id=0B3gersZ2cHIxRm5PMWRoTkdHdHc"
 
#COCO class labels for MobileNet-SSD
MOBILENET_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat",
    "bottle", "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]
 
COLORS = [
    (0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0),
    (0, 255, 255), (255, 0, 255), (128, 255, 0), (255, 128, 0)
]
 
 
def download_mobilenet_models():
    """Download MobileNet-SSD model files if not present"""
    import urllib.request
 
    if not os.path.exists(MOBILENET_PROTOTXT):
        print("Downloading MobileNet-SSD prototxt...")
        try:
            urllib.request.urlretrieve(PROTOTXT_URL, MOBILENET_PROTOTXT)
            print("  ✔ prototxt downloaded")
        except Exception as e:
            print(f"  ✘ Could not download prototxt: {e}")
            print("  Manual download: https://github.com/chuanqi305/MobileNet-SSD")
            sys.exit(1)
 
    if not os.path.exists(MOBILENET_MODEL):
        print(f"\n MobileNet-SSD caffemodel not found.")
        print("Please download manually from:")
        print("https://github.com/chuanqi305/MobileNet-SSD")
        print(f"Place '{MOBILENET_MODEL}' in the same folder as this script.")
        sys.exit(1)

## **OCR**

In [4]:
 
def run_ocr(image_path: str, psm: int = 3, save_output: bool = True):
    """
    Full OCR pipeline following the DecodeLabs specification:
    Step 1 — Grayscale conversion   (collapse RGB → intensity matrix)
    Step 2 — Gaussian blur          (remove noise and artifacts)
    Step 3 — Adaptive thresholding  (binary decision, Otsu's method)
    Step 4 — pytesseract OCR        (convolutional + LSTM pipeline)
    """
    import cv2
    import numpy as np
    import pytesseract
 
    print("PATH 1: OCR — Optical Character Recognition")
 
    #Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"  ✘ Could not load image: {image_path}")
        return
 
    h, w = img.shape[:2]
    print(f"\n  Image loaded   : {image_path}")
    print(f"  Dimensions     : {w} × {h} px  ({w * h:,} data points)")
 
    # Step1 Grayscale conversion
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    print(f"\n  [1/4] Grayscale   : 3D RGB matrix → 1D intensity matrix ✔")
 
    #Step2 Gaussian blur 
    # Smooths micro-imperfections and artifact noise
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    print(f"[2/4] Gaussian blur applied (kernel 5×5) ")
 
    #Step3: Adaptive thresholding (Otsu's method) 
    _, thresh = cv2.threshold(
        blur, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    print(f"  [3/4] Adaptive thresholding (Otsu's method) ✔")
 
    #Step4: pytesseract OCR
    config = f"--oem 3 --psm {psm}"
    raw_text = pytesseract.image_to_string(thresh, config=config)
    clean_text = raw_text.strip()
 
    print(f"  [4/4] pytesseract OCR  (PSM mode: {psm}) ✔")
 
    #Get confidence data
    data = pytesseract.image_to_data(
        thresh, config=config,
        output_type=pytesseract.Output.DICT
    )
    confidences = [int(c) for c in data["conf"] if str(c).isdigit() and int(c) >= 0]
    avg_conf = sum(confidences) / len(confidences) if confidences else 0
 
    #Display results
    print(f"\n  OCR Results")
    print(f"  Avg confidence : {avg_conf:.1f}%  {' PASS' if avg_conf >= 80 else 'BELOW 80% THRESHOLD'}")
    print(f"  Words detected : {len([w for w in data['text'] if w.strip()])}")
    print(f"\n   Extracted Text")
    print("  " + "\n  ".join(clean_text.split("\n")) if clean_text else "  (no text detected)")
 
    #Save processed images 
    if save_output:
        base = os.path.splitext(image_path)[0]
        cv2.imwrite(f"{base}_gray.png", gray)
        cv2.imwrite(f"{base}_thresh.png", thresh)
 
        # Draw word bounding boxes on original
        annotated = img.copy()
        n_boxes = len(data["text"])
        for i in range(n_boxes):
            if int(data["conf"][i]) >= 60:
                x, y, bw, bh = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
                cv2.rectangle(annotated, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
 
        cv2.imwrite(f"{base}_ocr_annotated.png", annotated)
        print(f"\n  Saved Outputs")
        print(f"  {base}_gray.png           ← grayscale")
        print(f"  {base}_thresh.png         ← thresholded")
        print(f"  {base}_ocr_annotated.png  ← word bounding boxes")
 
    return clean_text, avg_conf

## **Object Detection**

In [5]:
def run_object_detection(image_path: str, confidence_threshold: float = 0.80,save_output: bool = True):
    """
    Full object detection pipeline using MobileNet-SSD:
    Step 1 — Load image and construct 4D blob
    Step 2 — Forward pass through DNN
    Step 3 — Filter detections by 80% confidence gate
    Step 4 — Draw bounding boxes with labels and scores
    """
    import cv2
    import numpy as np
 
    print("Path 2: Object Detection — MobileNet-SSD")
 
    # Ensure model files exist
    download_mobilenet_models()
 
    #Load DNN model 
    print("\n  Loading MobileNet-SSD model...")
    net = cv2.dnn.readNetFromCaffe(MOBILENET_PROTOTXT, MOBILENET_MODEL)
    print(" Model loaded")
 
    #Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f" Could not load image: {image_path}")
        return
 
    h, w = img.shape[:2]
    print(f"\n  Image loaded   : {image_path}")
    print(f"  Dimensions     : {w} × {h} px  ({w * h:,} data points)")
 
    #Step 1: Blob construction
    blob = cv2.dnn.blobFromImage(
        cv2.resize(img, (300, 300)),
        scalefactor=0.007843,
        size=(300, 300),
        mean=127.5
    )
    print(f"\n  [1/4] Blob constructed (300×300, mean-subtracted) ")
 
    #Step 2: Forward pass 
    net.setInput(blob)
    detections = net.forward()
    print(f"  [2/4] DNN forward pass complete ")
 
    #Step 3: Filter by 80% confidence gate
    valid_detections = []
    for i in range(detections.shape[2]):
        confidence = float(detections[0, 0, i, 2])
        class_idx  = int(detections[0, 0, i, 1])
        if confidence >= confidence_threshold and class_idx < len(MOBILENET_CLASSES):
            box = detections[0, 0, i, 3:7] * [w, h, w, h]
            x1, y1, x2, y2 = box.astype(int)
            valid_detections.append({
                "label"     : MOBILENET_CLASSES[class_idx],
                "confidence": confidence,
                "box"       : (x1, y1, x2, y2)
            })
 
    print(f"  [3/4] Confidence gate (≥{confidence_threshold*100:.0f}%): "
          f"{len(valid_detections)} detections passed ✔")
 
    #Step 4: Draw bounding boxes 
    annotated = img.copy()
    for idx, det in enumerate(valid_detections):
        x1, y1, x2, y2 = det["box"]
        color = COLORS[idx % len(COLORS)]
        label = f"{det['label']}: {det['confidence']*100:.1f}%"
 
        # Draw box
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
 
        # Draw label background
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)[0]
        cv2.rectangle(annotated,
                      (x1, y1 - label_size[1] - 10),
                      (x1 + label_size[0] + 4, y1),
                      color, -1)
        cv2.putText(annotated, label, (x1 + 2, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
 
    print(f"  [4/4] Bounding boxes drawn ")
 
    #Display results 
    print(f"\n Detection Results")
    if valid_detections:
        for i, det in enumerate(valid_detections, 1):
            x1, y1, x2, y2 = det["box"]
            print(f"  #{i}  {det['label']:<15} "
                  f"Conf: {det['confidence']*100:.1f}%  "
                  f"Box: ({x1},{y1}) → ({x2},{y2})")
    else:
        print(f"  No objects detected above {confidence_threshold*100:.0f}% confidence threshold.")
        print("  Try lowering the threshold or using a different image.")
 
    #Save output
    if save_output:
        base = os.path.splitext(image_path)[0]
        output_path = f"{base}_detected.png"
        cv2.imwrite(output_path, annotated)
        print(f"\n  Saved Output")
        print(f"  {output_path}  ← annotated with bounding boxes")
 
    return valid_detections

## **Demo Mode**

In [6]:
def create_demo_image_ocr():
    import cv2
    import numpy as np
 
    img = np.ones((300, 600, 3), dtype=np.uint8) * 240
    texts = [
        ("DecodeLabs AI Project 4", (50, 60), 0.9, 2),
        ("Text Recognition Demo", (80, 110), 0.8, 2),
        ("OCR Pipeline: pytesseract", (50, 160), 0.7, 1),
        ("Confidence Threshold: 80%", (50, 200), 0.65, 1),
        ("Batch 2026", (200, 250), 0.75, 1),
    ]
    for text, pos, scale, thick in texts:
        cv2.putText(img, text, pos, cv2.FONT_HERSHEY_SIMPLEX, scale,(30, 30, 30), thick, cv2.LINE_AA)
 
    path = "demo_ocr_input.png"
    cv2.imwrite(path, img)
    return path
 

## **Main**

In [ ]:
def print_banner():
    print("Project 4: Image & Text Recognition")
    print("Optional Mastery Phase | Pre-trained Model Integration")
    print()
    print("  Path 1 — OCR Text Extraction   (pytesseract)")
    print("           Extract machine-readable text from images")
    print()
    print("  Path 2 — Object Detection      (MobileNet-SSD)")
    print("           Locate and label physical objects in images")
    print()
    print("  DEMO   — Run with synthetic test image (no file needed)")
    print()
 
 
def main():
    print_banner()
 
    print("  Select execution path:")
    print("  [1] OCR — Text Extraction")
    print("  [2] Object Detection — MobileNet-SSD")
    print("  [3] OCR Demo (generates test image automatically)")
    print()
 
    choice = input("  Enter choice (1 / 2 / 3): ").strip()
 
    if choice == "3" or choice.lower() == "demo":
        #DEMO MODE
        check_dependencies(1)
        print("\n  Generating synthetic OCR test image...")
        demo_path = create_demo_image_ocr()
        print(f"  Demo image saved → {demo_path}")
        run_ocr(demo_path, psm=6)
 
    elif choice == "1":
        # PATH 1: OCR
        check_dependencies(1)
        image_path = input("\n  Enter image path (or press Enter for demo): ").strip()
 
        if not image_path:
            print("  Generating demo image...")
            image_path = create_demo_image_ocr()
 
        print("\n  PSM modes:")
        print("    3  = Fully automatic (default, varied layouts)")
        print("    6  = Single block of text (book pages)")
        print("    7  = Single text line (number plates, headers)")
        print("   11  = Sparse scattered text (invoices)")
        psm_input = input("  Enter PSM mode [default=3]: ").strip()
        psm = int(psm_input) if psm_input.isdigit() else 3
 
        run_ocr(image_path, psm=psm)
 
    elif choice == "2":
        #PATH 2: OBJECT DETECTION
        check_dependencies(2)
        image_path = input("\n  Enter image path: ").strip()
 
        if not image_path:
            print("Path 2 requires a real image file.")
            print("Try: python project4_recognition.py")
            print("Then choose path 2 and provide an image path.")
            sys.exit(1)
 
        conf_input = input("  Confidence threshold [default=0.80]: ").strip()
        try:
            conf = float(conf_input) if conf_input else 0.80
        except ValueError:
            conf = 0.80
 
        run_object_detection(image_path, confidence_threshold=conf)
 
    else:
        print("  Invalid choice. Run the script again and enter 1, 2, or 3.")
 
    
    print("  Project 4 complete!")
    print("  Validation checklist:")
    print("  Library Integration    (pytesseract / cv2.dnn)")
    print("  Pre-Processing         (grayscale → blur → threshold)")
    print("  Accuracy Benchmarking  (≥80% confidence gate)")
    print("  Visual Confirmation    (annotated output saved)")
    
 
 
if __name__ == "__main__":
    main()

Project 4: Image & Text Recognition
Optional Mastery Phase | Pre-trained Model Integration

  Path 1 — OCR Text Extraction   (pytesseract)
           Extract machine-readable text from images

  Path 2 — Object Detection      (MobileNet-SSD)
           Locate and label physical objects in images

  DEMO   — Run with synthetic test image (no file needed)

  Select execution path:
  [1] OCR — Text Extraction
  [2] Object Detection — MobileNet-SSD
  [3] OCR Demo (generates test image automatically)

